In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl
import numpy as np
import gc
from deepctr.feature_column import SparseFeat, VarLenSparseFeat

# from preprocess import (
#     gen_data_set,
#     gen_model_input,
#     gen_data_set_sdm,
#     gen_model_input_sdm,
# )
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from tensorflow.python.keras import backend as K
from tensorflow.python.keras.models import Model

from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss, NegativeSampler
import sys

sys.path.append("../src/otto_recommender")
import utils
import deepmatch_preprocess

In [2]:
from tqdm.notebook import tqdm

tqdm.pandas()

In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 5
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch13_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    new_split = ""
    count_aid_trs = 60


config = Config()

In [4]:
dict_types = dict()
dict_types["clicks"] = 0
dict_types["carts"] = 1
dict_types["orders"] = 2

type_labels = pl.DataFrame(
    {"type": list(dict_types.keys()), "values": list(dict_types.values())}
)

In [5]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, new_split=False, count_aid_trs=60)
# X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
#     config, 
#     X_train, 
#     X_val, 
#     X_val_seq, 
#     ordinal_encoder, 
#     return_only_val=True, 
#     truncate_val=True
# )
pad = ordinal_encoder.unknown_value

# X_train = X_train.drop("count")
# X_train = X_train.drop("n_session_right")
# X_val = X_val.drop("count")
# X_val_seq = X_val_seq.drop("count")

CPU times: user 14min 44s, sys: 1min 44s, total: 16min 28s
Wall time: 3min 3s


In [6]:
pad

453724

In [7]:
%%time

X_train = (
    X_train.join(type_labels, on=config.col_type, how="left")
    .drop(columns=[config.col_type])
    .rename({"values": config.col_type})
)

CPU times: user 14.2 s, sys: 15.3 s, total: 29.5 s
Wall time: 2.8 s


In [8]:
%%time

X_val_seq = (
    X_val_seq.join(type_labels, on=config.col_type, how="left")
    .drop(columns=[config.col_type])
    .rename({"values": config.col_type})
)

CPU times: user 487 ms, sys: 192 ms, total: 679 ms
Wall time: 75.7 ms


In [9]:
# %%time

# X_train = (
#     X_train.groupby([config.col_user])
#     .agg(
#         [
#             pl.col(config.col_aid).list(),
#             pl.col(config.col_type).list(),

#         ]
#     ).filter(pl.col(config.col_aid).apply(lambda x: len(x)) >= 2)
# )

In [10]:
# %%time

# X_train = X_train.to_pandas()
# X_train[config.col_train_seq] = X_train[config.col_aid].progress_apply(lambda x: x[:-1])
# X_train[config.col_target] = X_train[config.col_aid].progress_apply(lambda x: x[-1])

# X_train['train_seq_type'] = X_train[config.col_type].progress_apply(lambda x: x[:-1])
# X_train['target_type'] = X_train[config.col_type].progress_apply(lambda x: x[-1])
# X_train = pl.from_pandas(X_train)

In [11]:
X_train["aid"].max()

453724

In [12]:
X_train["user"].max()

14571534

In [13]:
%%time

X_train = X_train.with_columns(
        [
            pl.col(config.col_aid)
            .cast(pl.Int32, strict=False)
            .alias(config.col_aid),
            pl.col(config.col_user)
            .cast(pl.Int32, strict=False)
            .alias(config.col_user),
            pl.col(config.col_type)
            .cast(pl.Int8, strict=False)
            .alias(config.col_type),
        ]
    )

CPU times: user 1.79 s, sys: 951 ms, total: 2.74 s
Wall time: 1.01 s


In [14]:
%%time

X_val_seq = X_val_seq.with_columns(
        [
            pl.col(config.col_aid)
            .cast(pl.Int32, strict=False)
            .alias(config.col_aid),
            pl.col(config.col_user)
            .cast(pl.Int32, strict=False)
            .alias(config.col_user),
            pl.col(config.col_type)
            .cast(pl.Int8, strict=False)
            .alias(config.col_type),
        ]
    )

CPU times: user 28.7 ms, sys: 23.5 ms, total: 52.2 ms
Wall time: 21.9 ms


In [15]:
X_train

user,aid,ts,n_session,num_action,type
i32,i32,i64,i32,i32,i8
14250687,222094,1662328738422,2,4,0
14250687,222094,1662328710275,2,5,0
14571534,66352,1662328709214,0,4,0
14571149,132357,1662328707619,0,4,0
14250687,222094,1662328701548,2,6,0
14565076,43952,1662328700064,0,4,0
14565076,36900,1662328695403,0,5,0
13029210,330137,1662328695309,1,4,0
14571519,258853,1662328691389,0,4,0


In [16]:
X_val_seq

user,aid,ts,n_session,num_action,type
i32,i32,i64,i32,i32,i8
14250687,222094,1662328738422,2,4,0
14250687,222094,1662328710275,2,5,0
14571534,66352,1662328709214,0,4,0
14571149,132357,1662328707619,0,4,0
14250687,222094,1662328701548,2,6,0
14565076,43952,1662328700064,0,4,0
14565076,36900,1662328695403,0,5,0
13029210,330137,1662328695309,1,4,0
14571519,258853,1662328691389,0,4,0


In [17]:
len(X_train[config.col_user].unique())

13366604

In [18]:
X_train[config.col_user].max()

14571534

In [19]:
%%time

SEQ_LEN = config.max_len
SEQ_LEN_short = 3
SEQ_LEN_prefer = 10
negsample = 0

# 1.Label Encoding for sparse features, and process sequence features with `gen_date_set` and `gen_model_input`
dict_lbes = dict()

feature_max_idx = dict()
feature_max_idx[config.col_type] = 2
feature_max_idx[config.col_aid] = pad
# for feature in [config.col_user]:
#     unknown_value = len(X_train[feature].unique())
#     lbe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=unknown_value)
#     X_train[feature] = lbe.fit_transform(X_train[feature].values.reshape(-1, 1)).astype('int32')
#     X_val_seq[feature] = lbe.transform(X_val_seq[feature].values.reshape(-1, 1)).astype('int32')
#     feature_max_idx[feature] = unknown_value
#     dict_lbes[feature] = lbe

CPU times: user 10 µs, sys: 1e+03 ns, total: 11 µs
Wall time: 18.4 µs


In [20]:
feature_max_idx

{'type': 2, 'aid': 453724}

In [21]:
# %%time

# import pickle

# with open(f'lbe_user_{config.count_aid_trs}.pickle', 'wb') as f:
#     pickle.dump(dict_lbes[config.col_user], f, protocol=4)

In [22]:
%%time

import pickle

with open(f'lbe_item_{config.count_aid_trs}.pickle', 'wb') as f:
    pickle.dump(ordinal_encoder, f, protocol=4)

CPU times: user 0 ns, sys: 3.62 ms, total: 3.62 ms
Wall time: 3.39 ms


In [23]:
ordinal_encoder

OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=453724)

In [24]:
X_train.head()

user,aid,ts,n_session,num_action,type
i32,i32,i64,i32,i32,i8
14250687,222094,1662328738422,2,4,0
14250687,222094,1662328710275,2,5,0
14571534,66352,1662328709214,0,4,0
14571149,132357,1662328707619,0,4,0
14250687,222094,1662328701548,2,6,0


In [25]:
X_train["type"]

type
i8
0
0
0
0
0
0
0
0
0


In [26]:
col_ts = config.col_ts
col_item = config.col_aid
col_type = config.col_type
col_user = config.col_user

In [27]:
%%time

X_val_seq = X_val_seq.to_pandas()

val_set = deepmatch_preprocess.gen_data_set(X_val_seq, seq_max_len=config.max_len, pad=pad, mode='val')

import pickle

with open(f'val_set_dssm_{config.count_aid_trs}.pickle', 'wb') as f:
    pickle.dump(val_set, f, protocol=4)

100%|████████████████████████████████| 466825/466825 [00:44<00:00, 10481.40it/s]


7
CPU times: user 47.8 s, sys: 676 ms, total: 48.5 s
Wall time: 48.2 s


In [28]:
len(val_set)

466825

In [29]:
val_set[0]

(12899780, 142193, 1, [279097], 1, [0], 0)

In [30]:
X_val_seq[X_val_seq["user"] == 12899780]

,user,aid,ts,n_session,num_action,type
3707354,12899780,279097,1661724000378,0,5,0
3707178,12899780,142193,1661724058352,0,4,0


In [31]:
del val_set, X_val_seq, X_val
gc.collect()

165

In [32]:
%%time

X_train = X_train.to_pandas()

train_set = deepmatch_preprocess.gen_data_set(X_train, seq_max_len=config.max_len, pad=pad, mode='train')

100%|█████████████████████████████| 13366604/13366604 [33:17<00:00, 6690.70it/s]


7
CPU times: user 34min 7s, sys: 1min 35s, total: 35min 43s
Wall time: 35min 17s


In [33]:
del X_train
gc.collect()

190

In [34]:
%%time

import pickle

with open(f'train_set_dssm_{config.count_aid_trs}.pickle', 'wb') as f:
    pickle.dump(train_set, f, protocol=4)

CPU times: user 2min 14s, sys: 23.2 s, total: 2min 37s
Wall time: 2min 38s


In [35]:
stop

NameError: name 'stop' is not defined

In [ ]:
data[data["user"] == 0].head(15)

In [ ]:
    # train_uid = np.array([line[0] for line in train_set])
    # train_iid = np.array([line[1] for line in train_set])
    # train_label = np.array([line[2] for line in train_set])
    # short_train_seq = [line[3] for line in train_set]
    # prefer_train_seq = [line[4] for line in train_set]
    # train_short_len = np.array([line[5] for line in train_set])
    # train_prefer_len = np.array([line[6] for line in train_set])
    # short_train_seq_genres = np.array([line[7] for line in train_set])
    # prefer_train_seq_genres = np.array([line[8] for line in train_set])

In [ ]:
train_set[13]

In [ ]:
item_id_types_map[954978]

In [ ]:
%%time

sparse_features = [
    "user",
    "aid",
    "type",
]

SEQ_LEN = 10
negsample = 0

feature_max_idx = dict()
feature_lbe = dict()
for feature in sparse_features:
    lbe = LabelEncoder()
    data[feature] = lbe.fit_transform(data[feature]) + 1
    feature_max_idx[feature] = data[feature].max() + 1
    feature_lbe[feature] = lbe

In [ ]:
feature_max_idx

In [ ]:
%%time

user_profile = data[["user", "aid", "type"]].drop_duplicates("user")

item_profile = data[["movie_id", "genres"]].drop_duplicates("movie_id")

user_profile.set_index("user_id", inplace=True)

user_item_list = data.groupby("user_id")["movie_id"].apply(list)

train_set, test_set = gen_data_set(data, SEQ_LEN, negsample)

train_model_input, train_label = gen_model_input(train_set, user_profile, SEQ_LEN)
test_model_input, test_label = gen_model_input(test_set, user_profile, SEQ_LEN)

In [ ]:
%%time

embedding_dim = 32

user_feature_columns = [SparseFeat('user_id', feature_max_idx['user_id'], 16),
                        SparseFeat("gender", feature_max_idx['gender'], 16),
                        SparseFeat("age", feature_max_idx['age'], 16),
                        SparseFeat("occupation", feature_max_idx['occupation'], 16),
                        SparseFeat("zip", feature_max_idx['zip'], 16),
                        VarLenSparseFeat(SparseFeat('hist_movie_id', feature_max_idx['movie_id'], embedding_dim,
                                                    embedding_name="movie_id"), SEQ_LEN, 'mean', 'hist_len'),
                        VarLenSparseFeat(SparseFeat('hist_genres', feature_max_idx['genres'], embedding_dim,
                                embedding_name="genres"), SEQ_LEN, 'mean', 'hist_len'),
                        ]

item_feature_columns = [SparseFeat('movie_id', feature_max_idx['movie_id'], embedding_dim),
                        SparseFeat('genres', feature_max_idx['genres'], embedding_dim)
                       ]

from collections import Counter
train_counter = Counter(train_model_input['movie_id'])
item_count = [train_counter.get(i,0) for i in range(item_feature_columns[0].vocabulary_size)]
sampler_config = NegativeSampler('inbatch', num_sampled=255, item_name="movie_id", item_count=item_count)

In [ ]:
%%time

import tensorflow as tf
if tf.__version__ >= '2.0.0':
    tf.compat.v1.disable_eager_execution()
else:
    K.set_learning_phase(True)
    
model = DSSM(user_feature_columns, item_feature_columns,user_dnn_hidden_units=(128,64, embedding_dim),
             item_dnn_hidden_units=(64, embedding_dim,),loss_type='softmax',sampler_config=sampler_config)

model.compile(optimizer="adam", loss=sampledsoftmaxloss)

history = model.fit(train_model_input, train_label,  # train_label,
                    batch_size=256, epochs=3, verbose=1, validation_split=0.0, )

# 4. Generate user features for testing and full item features for retrieval
test_user_model_input = test_model_input
all_item_model_input = {"movie_id": item_profile['movie_id'].values,"genres": item_profile['genres'].values}

user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)

user_embs = user_embedding_model.predict(test_user_model_input, batch_size=2 ** 12)
# user_embs = user_embs[:, i, :]  # i in [0,k_max) if MIND
item_embs = item_embedding_model.predict(all_item_model_input, batch_size=2 ** 12)

print(user_embs.shape)
print(item_embs.shape)